### Mouse brain - ligand-receptor communication in regions, subregions and between cell types (mp_mouse_brain04)

Same analysis as **mp_lymphoid04_liana_regions.ipynb** (both notebooks are generated from one template by **_make_mp_liana_regions.py**; only the loading and parameter cells differ). The regions are those of **mp_mouse_brain01** (Leiden on the cell2location abundances of the paper map; 17 labelled regions) and the subregions those of **mp_mouse_brain03** (each region clustered on gene expression, own UMAP).

**Question**: which ligand-receptor (LR) pairs are spatially active **in each region / subregion**, and **between which cell types** do they act?

#### Libs: **src/libs**

- **single_cell_lib** - Class **SingleCell**, section **9. LIANA** (dataset-independent): spatial_weights, liana_bivariate, calc_df_lr_by_group, calc_celltype_expr, calc_df_celltype_lr, calc_df_group_celltype_links, liana_rank_aggregate_reference, calc_df_celltype_coloc, calc_df_ref_lr_support, calc_df_group_ccc + figures; **DEFAULT_PARAMS** (liana_*, lr_*, c2l_lr_*, ref_lr_*, coloc_*)
- **single_cell_neuro_lib** - Class **SingleCellNeuro(SingleCell)**: data loading, regions, signatures; **NEURO_PARAMS**

| section | content |
|---|---|
| 1 | spots, regions, subregions, cell2location abundances |
| 2 | spatial weights (Gaussian kernel per section, bandwidth in spot spacings) |
| 3 | LIANA bivariate: global Moran's R and local score of every LR pair |
| 4 | LR pairs enriched in each region and subregion |
| 5 | between cell types: cell2location-resolved sender -> receiver (whole tissue) |
| 6 | between cell types inside each region / subregion |
| 7 | reference LIANA (rank_aggregate) + cell-type co-localisation |
| 8 | consensus per region / subregion |
| 9 | tables |

**Data**: the paper's cell2location map (**c2l_paper_map/sp_with_clusters.h5ad**; 2 sections, ST8059048 mouse 1 and ST8059051 mouse 2; 5,396 spots; no per-cell-type layers), the 59 regression signatures (**snrna/regression_signatures_sc.h5ad**) and the snRNA reference (**snrna/all_cells_20200625.h5ad** + 59-type annotation, 40,532 nuclei). LR resource: LIANA **mouseconsensus** (**NEURO_PARAMS['liana_resource']**).

Figures and tables have the prefix **mb04_**.

### Concepts

- **What a Visium LR score can mean.** A spot (55 µm, 100 µm spacing) holds several to ~30 cells of several types. The bivariate local score says that ligand and receptor are co-expressed **in a neighbourhood** (~2 rings, ~200 µm), not that two cells touch or signal.

- **Regions vs subregions.** The same functions run on both grouping levels (**group** = region or subregion); every group is compared with the rest of the tissue (**AUROC** of the local score; 0.5 = as elsewhere) and by the fraction of significant high-high spots.
- **Statistics.** Neighbouring spots are not independent, so Mann-Whitney / permutation p-values are optimistic; groups and links are ranked by effect size (AUROC, score, z) and the cutoffs are parameters.
- **Between cell types.** Visium does not see cell types; cell2location does. The counts of each spot are split across cell types in proportion to abundance x reference signature (**calc_celltype_expr**; identical to the per-cell-type layers cell2location writes, checked on the lymph node map), the ligand is taken from the sender type and the receptor from the receiver type. Attribution therefore follows the reference: a gene absent from a reference type is never attributed to it.
- **Same-type links** (A -> A) appear wherever one type carries both ligand and receptor in the same spots (e.g. neurons of one layer); they are kept in the tables and left out of the link heatmaps.
- **Reference LIANA is not spatial.** rank_aggregate on the scRNA reference ranks source -> target pairs by expression; it is annotated with the spatial co-localisation of the two types (whole tissue and per group).
- **Uninformative pairs.** MHC / B2M and intracellular proteins listed as ligands in the resource (**lr_display_exclude**) are computed but not listed or counted as links.
- **Two sections, two mice.** The spatial kernel is built per section (sections are never connected); the bivariate and group statistics pool both sections.
- **Neuronal resource content.** Many top pairs of the mouse resource are synaptic / adhesion pairs between neurons of one type (Nptx1->Nptxr, Rtn4->Lingo1); intracellular 'ligands' (Gnas, Arf1, Rims1, ...) are computed but not listed (**lr_display_exclude**).
- **edge_LowQ** is excluded from the group analyses (**subregion_skip**).

### **interaction, score, n_lr**

These three columns come from the cell-type-resolved LR analysis in mb04, which runs on cell2location and produces the sender → receiver heatmaps and the links table. 

The definitions below are taken from **_calc_df_celltype_lr** and **_calc_df_group_celltype_links** in **single_cell_lib.py**.

- **interaction** is the name of the ligand–receptor pair written **ligand^receptor**. 
  - An underscore marks a complex. For example, **Sema3c^Nrp2_Plxna4** is Sema3c binding the Nrp2 + Plxna4 receptor complex.

  - A complex's expression is the minimum over its subunits, so it counts only where every subunit is present.
  - The same key is used in every mb04 table, including the LIANA bivariate tables, the region/subregion enrichment and the consensus tables, so tables can be joined on it.


- **score** measures how strongly ligand from cell type A meets receptor from cell type B in space (rows **celltype_lr_**):

  - **score[A→B] = mean over spots s of ( W̄·L_A )(s) × R_B(s)**

  - **How each spot's counts are split:** a spot's counts are divided among cell types in proportion to cell2location abundance × the reference signature, the same way cell2location builds its per-cell-type layers. Each type's share is converted to log1p CP10k.

  - **L_A:** the ligand expression attributed to sender A. It is smoothed over neighbouring spots with the row-normalised spatial weights W̄.
  - **R_B:** the receptor expression attributed to receiver B at the spot itself.
  - **Per-group table:** the mean is taken over the group's spots only. **score_tissue** is the same quantity over the whole section, and **ratio** = score / score_tissue. For example, in THAL, Nptx1^Nptxr from Ext_Thal_2 to Ext_Thal_2 has score 0.024 against 0.0037 tissue-wide, a ratio of 6.6.
  - **Units:** the score is a product of log-expression values, so it is relative. It can be compared between links of the same LR pair, but not between pairs, because highly expressed pairs get larger scores. That is why each panel of the sender → receiver heatmap has its own colour scale (0.12 for Nptx1^Nptxr, 0.016 for Sst^Sstr2).
  - **z / p:** these come from permuting the receptor spots: 200 permutations tissue-wide, 50 within a group. This null ignores spatial autocorrelation, so z is inflated. Treat z as something to rank by, not as a calibrated p-value.


- **n_lr** (in **mb04_celltype_links_by_group.tsv**) counts, for one group (region or subregion) and one sender → receiver link, how many LR pairs support that link. An LR pair counts when all of these hold:
  - the link is among the group's top 10 sender → receiver links for that pair (**c2l_lr_group_top_k=10**);

  - z ≥ 3 in the within-group permutation (**c2l_lr_min_z=3.0**);
  - ratio ≥ 1, meaning the score in the group is at least the tissue-wide score;
  - the pair is not removed by the display filter (**lr_display_exclude**).


Two related columns sit next to it. **sum_score** is the sum of those pairs' scores, and **lr** lists the first six pairs by score. For example, in AMY the link Ext_Amy_2 → Ext_Amy_2 has n_lr = 63, led by App^Aplp2, Psap^Sort1 and Rtn4^Lingo1. The links heatmap (**plotly_group_celltype_links**) colours each cell by n_lr.

One caveat on n_lr: it counts how many pairs support a link, not how strong the link is. Self links (**self_link=True**) tend to come out on top. The cause is the deconvolution, not biology: in spots where one cell type dominates, both the ligand and the receptor are assigned to that type. **exclude_self=True** in the plot shows only links between different types.

## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi
import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_neuro_lib import SingleCellNeuro, NEURO_PARAMS, NEURO_FILES
from libs.matplotlib_figure_style import apply_figure_style

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['liana', 'scanpy', 'anndata', 'plotly']})

## Parameters

All parameters are class defaults (**NEURO_PARAMS** / **DEFAULT_PARAMS**; the LIANA values are the same in mp_lymphoid04 and mp_mouse_brain04; resource **liana_resource** = 'mouseconsensus'); override with **SingleCellNeuro(..., params=dict(...))**.

Files: **NEURO_FILES['inputs']** + **NEURO_FILES['mb04']** (**analysis='mb04'**), read and written by the class; the subregions are those cached by mp_mouse_brain03. Every method that writes a table or a figure has **force** and **verbose** (default False): with **force=False** an existing table is read (the LIANA runs and the cell-type permutations are the slow steps) and an existing figure is kept.

In [ ]:
force=False
verbose=False

apply_figure_style()
scn = SingleCellNeuro(ROOT0, analysis='mb04')
ref_groupby, ref_symbol_col = scn.params['celltype_col'], scn.params['ref_symbol_col']   # reference cell types; symbol column
print(scn)
print({k: scn.params[k] for k in ['liana_resource', 'liana_bandwidth_spots', 'liana_nz_prop', 'lr_group_min_auroc', 'lr_display_exclude']})
scn.calc_df_files()

### 1. Spots, regions, subregions and cell2location abundances

**prepare_expression**: symbols as var_names (LIANA resources use symbols), raw counts in **layers['counts']**, X = log1p(CP10k). Subregions are recomputed with the parameters of **mp_mouse_brain03_subregions.ipynb** (**run_subregions** is deterministic; the subregion table of that notebook gives their status).

In [ ]:
force=False
verbose=False

adata_map = scn.load_paper_map('c2l_map')
df_regions = scn.calc_df_regions(adata_map)
scn.add_region_labels(df_regions, adata_map)                                   # obs['region_label']
df_sig = scn.signatures_from_regression('signatures')                          # genes (symbols) x 59 cell types
adata_ref = scn.read_h5ad('reference', set_self=False, verbose=verbose)
adata_ref, _ = scn.add_paper_annotation('reference_annotation', adata=adata_ref, verbose=verbose)   # obs['annotation_1'], annotated nuclei

df_abund = scn.calc_df_abundance(adata_map)                 # q05 cell abundance (cells per spot)
adata_expr = scn.prepare_expression(adata_map)
subs, _ = scn.run_subregions(adata_expr, force=False, verbose=verbose)   # obs['sub_region_id']: the cached subsets of mb03
celltypes = [c for c in df_abund.columns if c in df_sig.columns]

df_region_info = scn.read_tsv('region_info', verbose=verbose)
region_name = dict(zip(df_region_info.region_id, df_region_info.region_name))
df_sub03 = scn.read_tsv('subregion_table_mb03', verbose=verbose)
sub_supported = df_sub03.loc[df_sub03.status.str.startswith('supported'), 'sub_region_id'].tolist()

groups = {'region': scn._groups(adata_expr, 'region_label'), 'subregion': scn._groups(adata_expr, 'sub_region_id')}
print(adata_expr.shape, '|', len(celltypes), 'cell types |', groups['region'].nunique(), 'regions |',
      groups['subregion'].nunique(), 'subregions (', len(sub_supported), 'supported in 03 )')
print('sections:', scn.library_ids(adata_expr))

### 2. Spatial weights

Gaussian kernel **w = exp(-d² / 2bw²)** with **bw = liana_bandwidth_spots x spot spacing**, built **per section** (block-diagonal: spots of different sections are never neighbours), self weight 1, weights < **liana_cutoff** removed. Expressing the bandwidth in spot spacings makes the kernel the same for data at different pixel scales.

In [ ]:
force=False
verbose=False

df_w = scn.spatial_weights(adata_expr, force=force, verbose=verbose)
W = adata_expr.obsp['spatial_connectivities']
df_w

### 3. LIANA bivariate: spatially co-expressed LR pairs

**li.mt.bivariate** (resource **liana_resource**):

  - global **Moran's R** (bivariate spatial autocorrelation of ligand and receptor;
  
  - permutation p, BH FDR over all tested pairs) and the **local cosine** score of every pair at every spot, with categories (1 = high-high) and local p-values. 
  - Ligand and receptor (all complex subunits) in >= **liana_nz_prop** of the spots. Cached in **root_result** (force = True recomputes).

In [ ]:
force=False
verbose=False

df_inter, adata_lr = scn.liana_bivariate(adata_expr, force=force, verbose=verbose)     # + mb04_liana_bivariate_global.tsv
df_lr_sig = scn.lr_significant(df_inter)
print(f"LR pairs tested: {len(df_inter)}; Moran's R > 0 at FDR <= {scn.params['liana_fdr']}: {len(df_lr_sig)}")
df_lr_sig[['ligand', 'receptor', 'ligand_props', 'receptor_props', 'morans', 'morans_fdr']].head(15).round(3)

In [ ]:
force=False
verbose=False

lr_show = scn._lr_display(df_lr_sig.index)[:8]
for lib in scn.library_ids(adata_expr):
    fig = scn.plot_lr_spatial(adata_lr, adata_expr, lr_show, library_id=lib, ncols=4, figsize=(14,8),
                              fontsize_title=11, fontsize_suptitle=16, fontsize_colorbar=8,
                              fname='fig_lr_top', force=force, verbose=verbose)
    plt.show()

### 4. LR pairs enriched in each region and subregion

**calc_df_lr_by_group**: per group, the local score of every significant pair in the group vs the rest of the tissue: **AUROC**, mean local score in / out, fraction of high-high spots (local p < 0.05) in / out. **enriched** = AUROC >= **lr_group_min_auroc** and more high-high spots than elsewhere.

In [ ]:
force=False
verbose=False

df_lr_region = scn.calc_df_lr_by_group(adata_lr, groups['region'], df_lr_sig.index, fname='lr_by_region', force=force, verbose=verbose)
df_lr_sub = scn.calc_df_lr_by_group(adata_lr, groups['subregion'], df_lr_sig.index, fname='lr_by_subregion', force=force, verbose=verbose)

df_n = pd.DataFrame({'n_enriched': df_lr_region[df_lr_region.enriched].groupby('group').size()}) \
         .reindex(sorted(groups['region'].dropna().unique())).fillna(0).astype(int)
df_n['region_name'] = df_n.index.map(region_name)
df_n['top_lr'] = scn.calc_df_lr_group_top(df_lr_region)
df_n

In [ ]:
force=False
verbose=False

fig = scn.plotly_lr_group_heatmap(df_lr_region, n_per_group=3, group_names=region_name, height=900, width=1800,
                                  fontsize_title=22, fontsize_xticks=16, fontsize_yticks=16, fontsize_colorbar=16,
                                  fontsize_hover=16, fname='fig_lr_region_hm', 
                                  force=force, verbose=verbose)
fig.show()

The top enriched pair of each region on the tissue (region outlined in cyan)



In [ ]:
force=False
verbose=False

top1 = df_lr_region[df_lr_region.enriched & df_lr_region.interaction.isin(set(scn._lr_display(df_lr_region.interaction.unique())))] \
         .groupby('group').head(1)
reg_lab = adata_expr.obs['region_label'].astype(str).values
for lib in scn.library_ids(adata_expr):
    fig = scn.plot_lr_spatial(adata_lr, adata_expr, list(top1.interaction), library_id=lib, ncols=4, figsize=(18, 8),
                              titles=[f"{g}: {i} (AUROC {a:.2f})" for g, i, a in zip(top1.group, top1.interaction, top1.auroc)],
                              outlines=[reg_lab == g for g in top1.group],
                              fontsize_title=12, fontsize_suptitle=16, fontsize_colorbar=11,
                              fname='fig_lr_regions', force=force, verbose=verbose)
    plt.show()

Subregions: the **supported** subregions of the 03 notebook (spatially coherent, own signature).

In [ ]:
force=False
verbose=False

df_n_sub = df_lr_sub[df_lr_sub.group.isin(sub_supported)]
print(f"{df_n_sub[df_n_sub.enriched].group.nunique()} of {len(sub_supported)} supported subregions have >= 1 enriched LR pair")
fig = scn.plotly_lr_group_heatmap(df_lr_sub, n_per_group=2, groups=[g for g in sub_supported if g in set(df_lr_sub.group)],
                                  height = 1400, width = 1900,
                                  fontsize_title=22, fontsize_xticks=15, fontsize_yticks=15, fontsize_colorbar=14,
                                  fontsize_hover=14, fname='fig_lr_sub_hm', force=force, verbose=verbose)
fig.show()

### 5. Between cell types: cell2location-resolved sender -> receiver (whole tissue)

**calc_celltype_expr**: counts of every spot split across cell types (abundance x signature), log1p(CP10k). **calc_df_celltype_lr** for the **lr_top_n** significant pairs with the highest Moran's R:

**score[A, B] = mean over spots of (W_norm · L_A) x R_B** - the ligand carried by sender A in the spot's neighbourhood times the receptor carried by receiver B in the spot. Null: receptor spots permuted (**c2l_lr_n_perms**) -> z. **score_share** = share of the pair's total over all A, B.

In [ ]:
force=False
verbose=False

df_lr_top = df_lr_sig.head(scn.params['lr_top_n'])[['ligand', 'receptor']]
genes = sorted({s for e in list(df_lr_top.ligand) + list(df_lr_top.receptor) for s in e.split('_')})
E, genes_E, celltypes_E = scn.calc_celltype_expr(adata_expr, df_abund, df_sig, genes)
print('E (spots x cell types x genes):', E.shape, '| LR genes without signature:', sorted(set(genes) - set(genes_E)))

df_ct, df_group_ct = scn.calc_df_celltype_lr(E, genes_E, celltypes_E, df_lr_top, W, groups=groups, force=force, verbose=verbose)

df_best = df_ct[df_ct.rank_in_lr == 1].set_index('interaction')
df_best = df_best.loc[[i for i in df_lr_top.index if i in df_best.index]]

print(df_ct.interaction.nunique(), 'LR pairs x', len(celltypes_E), 'x', len(celltypes_E), 'sender-receiver')
df_best[['sender', 'receiver', 'score', 'score_share', 'z']].head(15).round(3)

In [ ]:
force=False
verbose=False

lr_panels = scn._lr_display(df_best.index)[:6]
fig = scn.plot_sender_receiver_heatmaps(df_ct, lr_panels, n_top=10, ncols=3, panel_size=(8, 7.0),
                                        fontsize_title=18, fontsize_ticks=14,
                                        fontsize_label=14, fontsize_colorbar=12, fname='fig_sender_recv', force=force, verbose=verbose)
plt.show()

### 6. Between cell types inside each region / subregion

The same score over the spots of each group; receptor spots permuted **within** the group (**c2l_lr_group_n_perms**) -> z; **ratio** = group score / whole-tissue score. **calc_df_group_celltype_links**: per group and sender -> receiver, the number of LR pairs with z >= **c2l_lr_min_z** and ratio >= 1.

In [ ]:
force=False
verbose=False

df_links = scn.calc_df_group_celltype_links(df_group_ct, force=force, verbose=verbose)
lev = df_group_ct.drop_duplicates('group').set_index('group').level
df_links_reg = df_links[df_links.group.map(lev) == 'region']
df2 = df_links_reg[~df_links_reg.self_link].groupby('group').head(2)[['group', 'link', 'n_lr', 'lr']]

print(df2.shape)
df2

In [ ]:
fig = scn.plotly_group_celltype_links(df_links_reg, n_links=30, group_names=region_name, groups=sorted(df_links_reg.group.unique()),
                                     fontsize_title=22, fontsize_xticks=10, fontsize_yticks=12, fontsize_colorbar=14,
                                     fontsize_hover=13, fname='fig_links_region', force=force, verbose=verbose)
fig.show()

In [ ]:
df_links_sub = df_links[df_links.group.isin(sub_supported)]
fig = scn.plotly_group_celltype_links(df_links_sub, n_links=30, groups=[g for g in sub_supported if g in set(df_links_sub.group)],
                                     fontsize_title=22, fontsize_xticks=10, fontsize_yticks=12, fontsize_colorbar=14,
                                     fontsize_hover=13, fname='fig_links_sub', force=force, verbose=verbose)
fig.show()

### 7. Reference LIANA (rank_aggregate) + cell-type co-localisation

**liana_rank_aggregate_reference**: rank_aggregate (CellPhoneDB, Connectome, log2FC, NATMI, SCA, CellChat-like) on the reference, up to **ref_lr_cells_per_type** cells per type. Non-spatial, so each source -> target is annotated with **co-localisation** (**calc_df_celltype_coloc**): Pearson r between A at a spot and the kernel-weighted B around it (whole tissue), and each group's share of it. Specific and strong = specificity_rank <= **ref_lr_spec_rank** and magnitude_rank <= **ref_lr_mag_rank**; **spatially supported** = co-localisation r >= **coloc_min** or group share >= **coloc_group_min**.

In [ ]:
df_liref = scn.liana_rank_aggregate_reference(adata_ref, ref_groupby, symbol_col=ref_symbol_col, force=force, verbose=verbose)
df_coloc, coloc_region = scn.calc_df_celltype_coloc(df_abund.loc[adata_expr.obs_names, celltypes_E], W, groups['region'])
df_ref_sup = scn.calc_df_ref_lr_support(df_liref, df_coloc, coloc_region, force=force, verbose=verbose)
print(f"reference interactions: {len(df_liref)}; specific & strong: {len(df_ref_sup)}; spatially supported: {int(df_ref_sup.spatially_supported.sum())}")
df_ref_sup[df_ref_sup.spatially_supported & df_ref_sup.listed][['source', 'target', 'interaction', 'magnitude_rank', 'specificity_rank',
                                            'coloc', 'best_group', 'coloc_group']].head(15).round(3)

In [ ]:
force=False
verbose=False

fig = scn.plotly_coloc_heatmap(df_coloc, height=1600, width=1700,
                               fontsize_title=22, fontsize_ticks=16, fontsize_colorbar=14,
                               fontsize_hover=14, force=force, verbose=verbose)
fig.show()

### 8. Consensus per region / subregion

**calc_df_group_ccc**: for each group, its top enriched LR pairs (section 4) with the best cell2location sender -> receiver of the pair **in the group** (section 6: z, ratio vs tissue), and the reference: **ref_same_link** = rank_aggregate finds the same LR between that sender and receiver (specific and strong), **ref_links** = best spatially supported reference source -> target of the pair.

In [ ]:
df_ccc_region = scn.calc_df_group_ccc(df_lr_region, df_group_ct[df_group_ct.level == 'region'], df_ref_sup,
                                     group_names=region_name, fname='ccc_by_region', force=force, verbose=verbose)
print('LR pairs with a cell-type link in the group:', int((df_ccc_region.c2l_sender.fillna('') != '').sum()), 'of', len(df_ccc_region),
      '| same link in the reference:', int(df_ccc_region.ref_same_link.sum()))
df_ccc_region.round(2)

In [ ]:
df_ccc_sub = scn.calc_df_group_ccc(df_lr_sub[df_lr_sub.group.isin(sub_supported)],
                                  df_group_ct[df_group_ct.level == 'subregion'], df_ref_sup, n=3,
                                  fname='ccc_by_subregion', force=force, verbose=verbose)
df_ccc_sub.round(2)

### 9. Tables

Written by the methods (NEURO_FILES["mb04"], root_result, prefix **mb04_**): spatial weights, bivariate global table, LR by region / subregion, cell-type LR (tissue, by group), cell-type links by group, reference support, consensus by region / subregion; caches: bivariate h5ad, reference rank_aggregate.

### Notes for interpretation

- **AUROC** compares a group with the rest of the tissue; a pair can be enriched in a group only because its producers are concentrated there (composition), which is the point of the cell-type sections.
- **Sender -> receiver** calls follow the reference signatures x cell2location abundances; they say which types *can* carry ligand and receptor *where*, not that they signal.
- **z** values ignore spatial autocorrelation and are large for any co-localised pair; compare links by score, ratio and n_lr.
- **Parametrisation**: the same LIANA parameter values run unchanged on the lymph node (one section, human, 34 types) and the mouse brain (two sections, mouse, 59 types); only the resource, the files and the region settings differ.